# Formula 1 Race Results Analysis

Python, Pandas, and Matplotlib analysis of Formula 1 race results.


## 1. Load Dataset


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)


In [ ]:
df = pd.read_csv('f1_results.csv')
print('CSV file loaded successfully.')
df.head()


## 2. Preprocess Dataset


In [ ]:
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df = df.sort_values(by='date')

missing_values = df.isnull().sum()
print('Missing values in each column:')
print(missing_values)


## 3. First and Last Race Year


In [ ]:
first_year = df['year'].min()
last_year = df['year'].max()

print('First year:', first_year)
print('Last year:', last_year)


## 4. Distribution of Races by Month


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df['month'], bins=12, edgecolor='black')
plt.xlabel('Month')
plt.ylabel('Number of Races')
plt.title('Distribution of F1 Races by Month')
plt.xticks(range(1, 13))
plt.tight_layout()
plt.show()


## 5. Top Winning Drivers


In [ ]:
driver_wins_df = df['winner_name'].value_counts().reset_index()
driver_wins_df.columns = ['Driver Name', 'Number of Wins']
driver_wins_df = driver_wins_df.sort_values(by='Number of Wins', ascending=False)

print('Top 10 drivers by wins:')
display(driver_wins_df.head(10))


## 6. Most Successful Teams


In [ ]:
team_wins_df = df['team'].value_counts().reset_index()
team_wins_df.columns = ['Team', 'Number of Wins']
team_wins_df = team_wins_df.sort_values(by='Number of Wins', ascending=False)

print('Top 10 teams by wins:')
display(team_wins_df.head(10))


## 7. Driver(s) With the Most Wins in a Single Season


In [ ]:
wins_per_season = (
    df.groupby(['year', 'winner_name'])
    .size()
    .reset_index(name='Number of Wins')
)

max_wins_in_season = wins_per_season['Number of Wins'].max()
top_driver_season = wins_per_season[wins_per_season['Number of Wins'] == max_wins_in_season]

print('Driver(s) with the most wins in a single season:')
display(top_driver_season)


## 8. Grand Prix Locations With the Most Races


In [ ]:
races_by_country_df = df['grand_prix'].value_counts().reset_index()
races_by_country_df.columns = ['Country (Grand Prix)', 'Number of Races']

print('All Grand Prix race counts:')
display(races_by_country_df)

print('Top 10 Grand Prix locations by number of races:')
display(races_by_country_df.head(10))


## 9. Most Successful Drivers in Top 3 Grand Prix Locations


In [ ]:
top_3_countries = df['grand_prix'].value_counts().head(3).index
top_countries_df = df[df['grand_prix'].isin(top_3_countries)]

driver_wins_top_countries = top_countries_df['winner_name'].value_counts().reset_index()
driver_wins_top_countries.columns = ['Driver Name', 'Number of Wins']

print('Driver wins in top 3 Grand Prix locations:')
display(driver_wins_top_countries)


## 10. First Win Year for 3 Random Drivers


In [ ]:
random_drivers = df['winner_name'].drop_duplicates().sample(3, random_state=42)

first_wins_df = (
    df[df['winner_name'].isin(random_drivers)]
    .groupby('winner_name')['year']
    .min()
    .reset_index()
)
first_wins_df.columns = ['Driver Name', 'First Win Year']

print('First win year for 3 random drivers:')
display(first_wins_df)


## 11. Driver With the Longest Gap Between First and Last Win


In [ ]:
driver_years_df = (
    df.groupby('winner_name')['year']
    .agg(['min', 'max'])
    .reset_index()
)
driver_years_df['Year Difference'] = driver_years_df['max'] - driver_years_df['min']

max_year_diff = driver_years_df['Year Difference'].max()
longest_gap_driver = driver_years_df[driver_years_df['Year Difference'] == max_year_diff]

print('Driver(s) with the longest gap between first and last win:')
display(longest_gap_driver)


## 12. Win Trend of the Most Successful Driver


In [ ]:
most_successful_driver = df['winner_name'].value_counts().idxmax()
driver_data = df[df['winner_name'] == most_successful_driver]
wins_per_year = driver_data.groupby('year').size()

plt.figure(figsize=(10, 5))
plt.plot(wins_per_year.index, wins_per_year.values, marker='o')
plt.title(f'Trend of Wins Over Time for {most_successful_driver}')
plt.xlabel('Year')
plt.ylabel('Number of Wins')
plt.grid(True)
plt.tight_layout()
plt.show()


## 13. Win Probability by Decade


In [ ]:
df['decade'] = (df['year'] // 10) * 10

decade_total_races = df.groupby('decade').size().reset_index(name='Total Races')
driver_decade_wins = df.groupby(['decade', 'winner_name']).size().reset_index(name='Wins')

final_table = pd.merge(driver_decade_wins, decade_total_races, on='decade')
final_table['Win Probability'] = final_table['Wins'] / final_table['Total Races']
final_table = final_table.sort_values(['decade', 'Win Probability'], ascending=[True, False])

final_table.to_excel('question_13_win_probability_by_decade.xlsx', index=False)

print('Win probability by decade saved to Excel file successfully.')
display(final_table.head(10))
